<a href="https://colab.research.google.com/github/mbudisic/AIE6/blob/main/09_Finetuning_Embeddings/Fine_tuning_Embedding_Models_for_RAG_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Embeddings for RAG on Specific Data

As we start our "fine-tuning" week, we'll start with the lowest hanging improvement one can do for RAG - which is:

Fine-tuning embeddings!

- 🤝 Breakout Room #1:
  - Task 1: Dependencies and Boilerplate
  - Task 2: Loading Data
  - Task 3: Constructing a Fine-tuning Dataset
  - Task 4: Fine-tuning `snowflake-arctic-embed-l`
  - Task 5: Evaluating our Retriever



#### Basic Overview of Fine-tuning Embeddings

In essence, what we want to do when we fine-tune our embedding models is very simple:

```
Move the embeddings for questions relating to a document
closer together with that document
```

We can think of fine-tuning our embedding models as follows:

1) We have some pair of text items that *should* be closer together
  - `Question`, `Document` pairs
  - EX: `Who drives the bus?`, `The bus was driven by Kyle, the Bus Driver`.

2) We use these pairs as labeled data to fine-tune our embedding model.

The process of training helps the model more accurately associate our questions with the correct documents.

##### ❓ Question #1:

Describe the nuance between using Q&D pairs to train the embedding model vs. inter-document pairs/related sentences.

What caveats does this approach have? Are there any special considerations for what kind of Q's we should use?


##### ❗ Answer #1:

Training on "related sentences" encourages the model to stick to the theme
while generating text. That's likely essential for training the foundation model
as it would be useful for any application.

In the context of fine-tuning or RAG, it may be more important in situations where
the queries themselves are more generic, e.g., "Summarize this paper".

Training on Q/D pairs aims at a subtly different task - connecting the
semantic context of the query with the appropriate spot in the document.
This makes it more likely that the response will address the question asked by
the user when that question itself contains some relation to a specific context
in the document database.

As for caveats, perhaps we are in danger of disconnecting pieces of context by accident.
Since an explicit query-document will be used both as a positive example
and a negative (against all other rows in the document database), I would guess
it is important to maintain a chunk-overlap (e.g. two sentences per chunk,
with one-sentence overlap) in order to maintain context connection.

There may be additional danger that the real user queries will be substantially
different than autogenerated queries, e.g., use acronyms or information that is
not contained in the context. This may require adding context documents that
would help with "translating" queries from user-language to document-language.

Finally, I am not so sure about how successful the use of distinct rows as
negative examples is. At the same, asking the question generator to generate
*negative* examples seems very challenging ahead of production.


## Task 1: Dependencies and Boilerplate

We'll set up our `nest_asyncio` so we can leverage async loops in our Notebook.

We'll also install the required libraries we'll be using today, and set up our OpenAI API key!

### Nest Asyncio

In [10]:
import nest_asyncio

nest_asyncio.apply()

### Install Dependencies

>> NOTE: You do not need to do these steps if you are running this notebook locally with `uv`.

In [11]:
!pip install -qU langchain_openai langchain_huggingface langchain_core langchain langchain_community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00


In [12]:
!pip install -qU faiss-cpu python-pptx==1.0.2 nltk==3.9.1 pymupdf dotenv beautifulsoup4 lxml

In [13]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()
from google.colab import userdata

def load_env_if_not_present(key_name, prompt_message):
  try:
    os.environ[key_name] = userdata.get(key_name)
  except userdata.SecretNotFoundError:
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)
  finally:
    print(f"{key_name} retrieved.")

load_env_if_not_present("OPENAI_API_KEY","Please enter your OpenAI API key!")


OPENAI_API_KEY retrieved.


### Provide OpenAI API Key

## Task 2: Loading Data

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [14]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [15]:
import os

if not os.path.exists("data/2023_llms.html"):
    !curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html
else:
    print("File data/2023_llms.html already exists, skipping download.")

File data/2023_llms.html already exists, skipping download.


In [16]:
if not os.path.exists("data/2024_llms.html"):
    !curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html
else:
    print("File data/2024_llms.html already exists, skipping download.")

File data/2024_llms.html already exists, skipping download.


In [17]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import BSHTMLLoader

path = "data/"
text_loader = DirectoryLoader(path, glob="*.html", loader_cls=BSHTMLLoader)

Next, we'll set up a classic naive chunking strategy as we only care that the documents get parsed into chunks that we can generate synthetic questions about.

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 750,
    chunk_overlap  = 20,
    length_function = len
)

Next we can load/split these documents as follows.

> NOTE: You may need to run this cell twice to get it to work.

In [19]:
training_documents = text_splitter.split_documents(text_loader.load())

In [20]:
len(training_documents)

102

Next, we're going to associate each of our chunks with a unique identifier.

In [21]:
import uuid
from typing import Set
id_set = set()

def unique_id(where:Set[str], prefix:str=""):
  """Create a unique ID with respect to the given set.

  (Optional) prefix is used to distinguish queries from documents"""
  id = prefix+str(uuid.uuid4())
  while id in where:
    id = prefix+str(uuid.uuid4())

  assert not(id == prefix)
  return id


for document in training_documents:
  doc_id = unique_id(id_set,"D-")
  id_set.add(doc_id)
  document.metadata["id"] = doc_id

Next, we'll simply use naive Python slicing to create a training, test, and validation set to prepare our data for the next step.

In [22]:
training_split_documents = training_documents[:len(training_documents) - 24]
validation_split_documents = training_documents[len(training_documents) - 24:102-12]
test_split_documents = training_documents[102-12:]

## Task 3: Constructing a Fine-tuning Dataset

Using the nodes we created above, we can finally start constructing a fine-tuning dataset utilizing OpenAI's `gpt-4.1-mini`

The basic idea here is straightforward enough:

1. We look at a document
2. We generate questions that could be answered by that node

This gives us a number of question/context pairs that we can use to fine-tune our Embeddings model.

In [23]:
from langchain_openai import ChatOpenAI

qa_chat_model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

We'll create a simple Question Generation prompt to query `gpt-4.1-mini` to generate Questions for each retrieved context.

In [24]:
from langchain_core.prompts import ChatPromptTemplate

qa_prompt = """\
Given the following context, you must generate questions based on only the provided context.

You are to generate {n_questions} questions which should be provided in the following format:

1. QUESTION #1
2. QUESTION #2
3. QUESTION #3
...

Each question should be a single line, and should not include any other text.

Context:
{context}
"""

qa_prompt_template = ChatPromptTemplate.from_template(qa_prompt)

We'll create a simple chain to query the LLM!

In [25]:
question_generation_chain = qa_prompt_template | qa_chat_model

There's a lot going on in this function - let's take a deeper look:

1. First, we provide a list of documents and a number of questions
2. We, for each document in our list, generate `n_questions` of questions.
3. We then associate those questions and contexts via a `UUID`.

> NOTE: The reason we're doing this `UUID` association is for ease of use later in the notebook.

##### 🏗️ Activity #1:

We have:

- Lists of `Documents` with the `metadata` field `id`.

We need:

- An object with key `id`, which have values `str` questions.
- An object with key `question_id`, which have values `List(str)` which will be a list of associated `context_id`.

An Example:

question_object:
```python
{
'b4b95fb6-f827-4454-aa5b-20e62733f172': 'What types of accessible formats are available for persons with disabilities?',
'df58ee4f-714c-419e-8324-94e5870574e2': 'How do accessible formats benefit persons with disabilities?',
'505fce8b-0e56-48de-a251-61027e396918': 'What are some of the risks associated with the increasing capabilities of AI systems that generate synthetic content?',
'8ff0ab33-60dc-4fee-8958-91bfb686aca8': 'Why is it important for providers of AI systems to embed technical solutions for marking and detecting synthetic content?'
}
 ```

 context_object:
 ```python
{
'b4b95fb6-f827-4454-aa5b-20e62733f172': ['dd75bf94-75f3-4603-8e4b-5522f6925638'],
'df58ee4f-714c-419e-8324-94e5870574e2': ['dd75bf94-75f3-4603-8e4b-5522f6925638'],
'505fce8b-0e56-48de-a251-61027e396918': ['ffe3893f-688c-48e8-90bd-7a9feb953d90'],
'8ff0ab33-60dc-4fee-8958-91bfb686aca8': ['ffe3893f-688c-48e8-90bd-7a9feb953d90'],
}
 ```

 As you can see, a piece of context can be associated with more than 1 question.

 The task is to write the Python function(s) to accomplish this task.

 Your function signature is provided below, along with the desired return values.

 > NOTE: You can make any modifications that you desire - assuming that you have the correct input and outputs.

Let's first take a look at the format we receive from the LLM.

In [26]:
test = question_generation_chain.invoke({"n_questions" : 5, "context" : """
Language models (LMs) have become ubiquitous in both NLP research
and in commercial product offerings. As their commercial
importance has surged, the most powerful models have become
closed off, gated behind proprietary interfaces, with important
details of their training data, architectures, and development
undisclosed. Given the importance of these details in
scientifically studying these models, including their biases and
potential risks, we believe it is essential for the research
community to have access to powerful, truly open LMs. To this
end, we have built OLMo, a competitive, truly Open Language
Model, to enable the scientific study of language models. Unlike
most prior efforts that have only released model weights and
inference code, we release OLMo alongside open training data and
training and evaluation code. We hope this release will empower
the open research community and inspire a new wave of innovation.
"""})

test.pretty_print()

================================== Ai Message ==================================

1. What has caused the most powerful language models to become closed off and proprietary?  
2. Why is it important for the research community to have access to powerful, truly open language models?  
3. What distinguishes OLMo from most prior efforts in releasing language models?  
4. What components are released alongside OLMo to support scientific study?  
5. What is the intended impact of releasing OLMo to the open research community?


Now, we need to extract the questions from that format:

In [27]:
import regex as re
def question_extract(raw:str):
    pattern = re.compile(r'^\d+\.\s+(.+)$', re.MULTILINE)
    return [match.strip() for match in re.findall(pattern, raw)]

question_extract(test.content)


['What has caused the most powerful language models to become closed off and proprietary?',
 'Why is it important for the research community to have access to powerful, truly open language models?',
 'What distinguishes OLMo from most prior efforts in releasing language models?',
 'What components are released alongside OLMo to support scientific study?',
 'What is the intended impact of releasing OLMo to the open research community?']

Let's see what we get if the format is wrong --- for handling edge-cases.

In [28]:
question_extract("Something without questions")

[]

In [29]:
import tqdm
import asyncio
from langchain.schema import Document
from typing import List
from warnings import warn

async def create_questions(documents: List[Document], n_questions: int):
  """Generate questions based on documents stored in a list

  Args:
      documents: List of Document objects containing text content to generate questions from
      n_questions: Number of questions to generate per document

  Returns:
      Tuple containing:
          - Dictionary mapping question IDs to question text
          - Dictionary mapping document IDs to lists of question IDs relevant to that document
  """
  questions = {}
  relevant_context = {}

  # Asynchronously generate questions for each document
  ## THIS MADE A HUGE DIFFERENCE IN SPEED
  all_questions_raw = await asyncio.gather( *(
      question_generation_chain.ainvoke({'context': doc.page_content, 'n_questions': n_questions})
      for doc in documents )
    )

  # Process document-questionlist pairs
  for doc,questions_raw  in zip(documents, all_questions_raw):

    doc_id = doc.metadata["id"]

    # populate the questions database
    question_list = question_extract(questions_raw.content)

    if len(question_list) == 0:
      warn(f"LLM didn't generate any questions based on {doc_id}.")
      continue

    for question in question_list:
        question_id = unique_id(questions.keys(), prefix="Q-")
        questions[question_id] = question
        if question_id not in relevant_context.keys() or len(relevant_context[question_id]) == 0:
          relevant_context[question_id] = [doc_id]
        else:
          relevant_context[question_id].append(doc_id)

  return questions, relevant_context

### REMOVE `await` IF NOT USING ASYNC (HINT: Use `async`)

We'll use the function to generate training, validation, and test data.

In [30]:
test_questions, test_relevant_contexts = await create_questions(test_split_documents, 2)
test_split_documents

[Document(metadata={'source': 'data/2023_llms.html', 'title': 'Stuff we figured out about AI in 2023', 'id': 'D-d4b77040-3668-47ad-b16f-ff11494c2e88'}, page_content='A lot of people are excited about AI agents—an infuriatingly vague term that seems to be converging on “AI systems that can go away and act on your behalf”. We’ve been talking about them all year, but I’ve seen few if any examples of them running in production, despite lots of exciting prototypes.\nI think this is because of gullibility.\nCan we solve this? Honestly, I’m beginning to suspect that you can’t fully solve gullibility without achieving AGI. So it may be quite a while before those agent dreams can really start to come true!\nCode may be the best application\nOver the course of the year, it’s become increasingly clear that writing code is one of the things LLMs are most capable of.'),
 Document(metadata={'source': 'data/2023_llms.html', 'title': 'Stuff we figured out about AI in 2023', 'id': 'D-e7aa0707-b716-4294

In [31]:
test_questions

{'Q-fdcf1b24-37e0-4e31-99f6-f54a80449a1e': 'What is the main reason given for the lack of AI agents running in production despite many prototypes?',
 'Q-48fb3160-7571-4de0-97e1-7e78c7fd876c': 'Why might achieving AGI be necessary to fully solve gullibility in AI agents?',
 'Q-06f0c0ac-7cd5-470e-ae70-f1e24a86d450': 'Why are the grammar rules of programming languages considered less complicated than those of natural languages?',
 'Q-4bd8bac3-c605-4e83-a08a-668c72356acf': 'What is one major weakness of large language models mentioned in the context?',
 'Q-c1a9299e-f00b-43f0-bd0f-862776813397': 'How does the ability of LLMs to execute and debug code reduce hallucination in code generation?',
 'Q-05e63c06-ed83-4a88-907f-88d139a1458e': "Why might software engineers feel threatened by ChatGPT's capability to write and correct code?",
 'Q-a25280ed-3299-4c27-a9e3-d3dacf198206': 'How can software engineers leverage their knowledge to utilize coding interns more effectively?',
 'Q-f86891c4-a402-4

In [32]:
test_relevant_contexts

{'Q-fdcf1b24-37e0-4e31-99f6-f54a80449a1e': ['D-d4b77040-3668-47ad-b16f-ff11494c2e88'],
 'Q-48fb3160-7571-4de0-97e1-7e78c7fd876c': ['D-d4b77040-3668-47ad-b16f-ff11494c2e88'],
 'Q-06f0c0ac-7cd5-470e-ae70-f1e24a86d450': ['D-e7aa0707-b716-4294-a678-f8df2f7a9204'],
 'Q-4bd8bac3-c605-4e83-a08a-668c72356acf': ['D-e7aa0707-b716-4294-a678-f8df2f7a9204'],
 'Q-c1a9299e-f00b-43f0-bd0f-862776813397': ['D-7e5da012-cd38-4e3a-b694-381083a4cdb3'],
 'Q-05e63c06-ed83-4a88-907f-88d139a1458e': ['D-7e5da012-cd38-4e3a-b694-381083a4cdb3'],
 'Q-a25280ed-3299-4c27-a9e3-d3dacf198206': ['D-79796aec-dc21-4baf-9f86-497ca677f23c'],
 'Q-f86891c4-a402-4852-9c49-eecfb278e076': ['D-79796aec-dc21-4baf-9f86-497ca677f23c'],
 'Q-2067fb51-f45f-42ec-8df6-a98833fa76e1': ['D-37d390fe-1b24-4991-97b9-d17f99c12d0a'],
 'Q-1a33b919-bc9d-4150-8af0-8c05a49e9b94': ['D-37d390fe-1b24-4991-97b9-d17f99c12d0a'],
 'Q-13f695e8-3ce4-4c10-8fc8-5c59abd4aaa1': ['D-153e2cad-ac5a-403b-8783-520e9a5bd190'],
 'Q-8fcaa8d8-d541-419a-a3be-a0f14ac94477': 

In [33]:
val_questions, val_relevant_contexts = await create_questions(validation_split_documents, 2)

In [34]:
training_questions, training_relevant_contexts = await create_questions(training_split_documents, 2)

### Reformating and Saving Datasets

Now, we can save our datasets for later use!

In [35]:
import json

training_corpus = {train_item.metadata["id"] : train_item.page_content for train_item in training_split_documents}

train_dataset = {
    "questions" : training_questions,
    "relevant_contexts" : training_relevant_contexts,
    "corpus" : training_corpus
}

with open("training_dataset.jsonl", "w") as f:
  json.dump(train_dataset, f)

In [36]:
validation_corpus = {val_item.metadata["id"] : val_item.page_content for val_item in validation_split_documents}

validation_dataset = {
    "questions" : val_questions,
    "relevant_contexts" : val_relevant_contexts,
    "corpus" : validation_corpus
}

with open("validation_dataset.jsonl", "w") as f:
  json.dump(validation_dataset, f)

In [37]:
train_corpus = {test_item.metadata["id"] : test_item.page_content for test_item in test_split_documents}

test_dataset = {
    "questions" : test_questions,
    "relevant_contexts" : test_relevant_contexts,
    "corpus" : train_corpus
}

with open("test_dataset.jsonl", "w") as f:
  json.dump(test_dataset, f)

## Task 4: Fine-tuning `snowflake-arctic-embed-l`

Now that we have a dataset, let's grab a `sentence-transformers` Embeddings model!

We'll be using Snowflake's [`snowflake-arctic-embed-l`](https://huggingface.co/Snowflake/snowflake-arctic-embed-l) as a base embeddings model.

It is a well performing embeddings model by itself, but there's a lot of very specific domain terms and vocabulary in our courpus - so lets fine-tune it and see what that can do for us!

>> NOTE: Skip installing dependencies if you are running this notebook locally.

In [38]:
!pip install -qU sentence_transformers datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.2.1 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 20.0.0 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
pylibcudf-cu12 25.2.1 requires p

In [39]:
from sentence_transformers import SentenceTransformer

model_id = "Snowflake/snowflake-arctic-embed-l"
model = SentenceTransformer(model_id)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/85.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

We'll grab some necessary imports from `sentence_transformers` and `torch`.

> NOTE: PyTorch (`torch`) is a popular machine learning library - while we don't go very deep into PyTorch it's an incredibly powerful and interesting library! Please read more about it [here](https://pytorch.org/tutorials/beginner/basics/intro.html)!

In [40]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from sentence_transformers import InputExample

We're using a toy batch size here to reflect the limited number of examples we have.

> NOTE: It is typical to use a much larger batch size (~64+), hardware permitting.

In [41]:
BATCH_SIZE = 10

Let's move our dataset into the expected format for training.

In [42]:
corpus = train_dataset['corpus']
queries = train_dataset['questions']
relevant_context = train_dataset['relevant_contexts']

examples = []
for query_id, query in queries.items():
    doc_id = relevant_context[query_id][0]
    text = corpus[doc_id]
    example = InputExample(texts=[query, text])
    examples.append(example)

Now we can create a `torch` `DataLoader`!

In [43]:
loader = DataLoader(
    examples, batch_size=BATCH_SIZE
)

Next up, we'll prepare our loss function!

Loss is an important part of training, fine-tuning, and more. If you want a deep dive on loss - you can check out our [event on loss!](https://www.youtube.com/watch?v=iB8FWR9aD5Q&t=8s).

The core loss we're using today is called `MultipleNegativesRankingLoss` - you can find more information [here](https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/losses/MultipleNegativesRankingLoss.py).

This is "wrapped" in `MatryoshkaLoss`, which you can read the implementation of [here](https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/losses/MatryoshkaLoss.py).

In [44]:
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss

matryoshka_dimensions = [768, 512, 256, 128, 64]
inner_train_loss = MultipleNegativesRankingLoss(model)
train_loss = MatryoshkaLoss(
    model, inner_train_loss, matryoshka_dims=matryoshka_dimensions
)

##### 🏗️ Activity #2:

Both of these losses sound "cool", but what are they - exactly - under the hood?

Why are these losses specifically doing? Please write a short summary of each loss.

> NOTE: This is a course focused on AI Engineering and the application of AI - looking for a hint? Try pasting the code (linked above) into ChatGPT/Claude to write the summary!

##### ❗Activity #2 answer:

`MultipleNegativesRankingLoss` provides an optimization cost based on association of a query with
context documents. It reads the paired query-document from the training set
as the "positive" example, and all other query-document pairings in the training set as "negative"
examples. In principle, one could include additional explicit negative pairings
but we do not use that here.

This will in turn become the "distance" beetween the embedding vectors corresponding
to queries and context documents.

We can think of the corresponding optimization as a problem where we have
points in a space (queries and context documents) and we know their
prescribed distances. The goal for the embedding model is to find coordinates
for each point such that the prescribed distances are obeyed as close as possible.
Moreover, we are training a "machine" for generating the coordinates in such a way
that for these given points, the above is true.

`MatryoshkaLoss` poses the additional question: what number of coordinates
are we allowed to tune? It basically says: think of the given list of coordinates
as an ordered list, e.g., (1,2,4).

We are training a 4-dimensional coordinate representation, but we form
the total loss by constraining that a certain number of parameters is zero

- 1: `(*,0,0,0)`
- 2: `(*,*,0,0)`
- 4: `(*,*,*,*)`

This is done by constraining which elements of the vector can be varied
during the gradient calculation. The losses for all these subproblems are summed
(with optional weights) which gives the final loss.

If weights are skipped and the example problem above is given,
the change in the first element incurs a loss from all three subproblems,
for the second element for two subproblems. So the impact of getting the coordinates
right is basically 3:2:1:1, meaning getting the first coordinate "wrong"
is 3x more dangerous than getting the last two coordinates wrong.

Consequently, when we (after training) reduce the dimension of the embedding
from 4 -> 2, we are effectively setting the last two coordinates = 0 for embedding representations,
therefore intentionally making an error in the last two coordinates.
Since the embedding model was trained to be more tolerant to errors in
last two coordinates, that is, more likely to preserve the prescribed inner-loss
distances when the last two coordinates are wrong, we effectively
get an embedding into a nested set of spaces where reducing the dimension
to a prescribed subdimension by throwing out coordinates is done in an optimal way.


Now we can set-up our evaluator.

> NOTE: Due to the formatting of our dataset - this is all we have to do!

In [45]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

corpus = validation_dataset['corpus']
queries = validation_dataset['questions']
relevant_context = validation_dataset['relevant_contexts']

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_context)

We'll train this model for 5 epochs, though you could increase this number if we had a significant amount more data.

In [46]:
EPOCHS = 10

It's training time!

> NOTE: We're manually defining a warm-up period here - this is just to provide a smooth ramp into our training!

In [47]:
import wandb

load_env_if_not_present("WANDB_API_KEY","Please enter your WANDB API key!")
load_env_if_not_present("HF_TOKEN","Please enter your HF token!")

wandb.init(mode="online",
               # Set the wandb entity where your project will be logged (generally your team name).
    entity="budisicm-virginia-commonwealth-university",
    # Set the wandb project where this run will be logged.
    project="Finetuning Embedding Models"
    )
wandb.run.name = wandb.run.id
wandb.run.save()


WANDB_API_KEY retrieved.
HF_TOKEN retrieved.


wandb: Currently logged in as: budisicm (budisicm-virginia-commonwealth-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Calling wandb.run.save without any arguments is deprecated.Changes to attributes are automatically persisted.


True

> NOTE: You may not see direct improvement during the training cycles - this is absolutely expected. We will verify performance later in the notebook.

In [48]:
from huggingface_hub import notebook_login

notebook_login()

In [49]:
warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path='finetuned_arctic_ft',
    show_progress_bar=True,
    evaluator=evaluator,
    evaluation_steps=50
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss,Cosine Accuracy@1,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Accuracy@10,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
16,No log,No log,0.791667,1.000000,1.000000,1.000000,0.791667,0.333333,0.200000,0.100000,0.791667,1.000000,1.000000,1.000000,0.923110,0.895833,0.895833
32,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.953866,0.937500,0.937500
48,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.953866,0.937500,0.937500
50,No log,No log,0.875000,1.000000,1.000000,1.000000,0.875000,0.333333,0.200000,0.100000,0.875000,1.000000,1.000000,1.000000,0.953866,0.937500,0.937500
64,No log,No log,0.958333,1.000000,1.000000,1.000000,0.958333,0.333333,0.200000,0.100000,0.958333,1.000000,1.000000,1.000000,0.984622,0.979167,0.979167
80,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.969244,0.958333,0.958333
96,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.969244,0.958333,0.958333
100,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389
112,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389
128,No log,No log,0.916667,1.000000,1.000000,1.000000,0.916667,0.333333,0.200000,0.100000,0.916667,1.000000,1.000000,1.000000,0.963789,0.951389,0.951389


In [50]:
load_env_if_not_present("HF_USER","Please enter your HuggingFace username!")

HF_USER retrieved.


In [51]:
import datetime
username = os.environ["HF_USER"]
## uncomment with specific model name to override push
# model_hf_name
try:
  model_hf_name
except:
  modelid_tag = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
  model_hf_name = f"{username}/snoflake-simon-{modelid_tag}"
finally:
  print(f"Pushing model to {model_hf_name}")
  model.push_to_hub(model_hf_name)

Pushing model to mbudisic/snoflake-simon-20250507161314


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

## Task 5: Evaluating our Retriever

Now that we have fine-tuned our retriever - let's see if it's worthwhile!

We'll start with some basic imports.

In [52]:
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.documents import Document

Now we'll define a function that will help us evaluate our retrieval process.

> NOTE: We're assuming 1 correct document in a "hit".

In [53]:
def evaluate_openai(
    dataset,
    embed_model,
    top_k=5,
    verbose=False,
):
  corpus = dataset['corpus']
  questions = dataset['questions']
  relevant_docs = dataset['relevant_contexts']
  documents = [Document(page_content=content, metadata={"id": doc_id}) for doc_id, content in corpus.items()]
  vectorstore = FAISS.from_documents(documents, embed_model)

  retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

  eval_results = []
  for id, question in tqdm.tqdm(questions.items()):
    retrieved_nodes = retriever.invoke(question)
    retrieved_ids = [node.metadata["id"] for node in retrieved_nodes]
    expected_id = relevant_docs[id][0]
    is_hit = expected_id in retrieved_ids
    eval_results.append({"id": id, "question": question, "expected_id": expected_id, "is_hit": is_hit})

  return eval_results

All that's left to do is evaluate, we'll evaluate our model against:

1. OpenAI's closed source `text-embedding-3-small`
2. The base non-fine-tuned version of `Snowflake/snowflake-arctic-embed-l`.

Let's see how it stacks up!

### `text-embedding-3-small`

In [54]:
te3_openai = OpenAIEmbeddings(model="text-embedding-3-small")
te3_results = evaluate_openai(test_dataset, te3_openai)

100%|██████████| 24/24 [00:20<00:00,  1.15it/s]


In [55]:
te3_results_df = pd.DataFrame(te3_results)

In [56]:
te3_hit_rate = te3_results_df["is_hit"].mean()
te3_hit_rate

np.float64(1.0)

### `Snowflake/snowflake-arctic-embed-l` (base)

In [57]:
from langchain_huggingface import HuggingFaceEmbeddings

huggingface_embeddings = HuggingFaceEmbeddings(model_name="Snowflake/snowflake-arctic-embed-l")
arctic_embed_m_results = evaluate_openai(test_dataset, huggingface_embeddings)

100%|██████████| 24/24 [00:00<00:00, 47.12it/s]


In [58]:
arctic_embed_m_results_df = pd.DataFrame(arctic_embed_m_results)

In [59]:
arctic_embed_m_hit_rate = arctic_embed_m_results_df["is_hit"].mean()
arctic_embed_m_hit_rate

np.float64(0.9583333333333334)

### `Snowflake/snowflake-arctic-embed-l` (fine-tuned)

In [60]:
finetune_embeddings = HuggingFaceEmbeddings(model_name="finetuned_arctic_ft")
finetune_results = evaluate_openai(test_dataset, finetune_embeddings)

Some weights of BertModel were not initialized from the model checkpoint at finetuned_arctic_ft and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 24/24 [00:00<00:00, 48.63it/s]


In [61]:
finetune_results_df = pd.DataFrame(finetune_results)

In [62]:
finetune_hit_rate = finetune_results_df["is_hit"].mean()
finetune_hit_rate

np.float64(1.0)

## Task 1: Vibe Checking the RAG Pipeline

We're going to use our RAG pipeline to vibe check on some common phrases now that we've modified it!

### Creating New Chunks

In order to try and evaluate our system more fairly, let's create new chunks that we will use to create our Vector Store.

In [63]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap  = 50,
    length_function = len
)

training_documents = text_splitter.split_documents(text_loader.load())

### Base Chain

We'll start by constructing our base chain, which will use the untrained retrieval model.

#### R - Retrieval

In [64]:
from langchain_community.vectorstores import FAISS

base_vectorstore = FAISS.from_documents(training_documents, huggingface_embeddings)
base_retriever = base_vectorstore.as_retriever(search_kwargs={"k": 6})

#### A - Augmented

In [65]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and a question, you must answer the question. If you do not know the answer, you must state that you do not know.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt_template = ChatPromptTemplate.from_template(RAG_PROMPT)

#### G - Generation

In [66]:
rag_llm =  ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0
)

#### RAG - LCEL RAG Pipeline

In [67]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

base_rag_chain = (
    {"context": itemgetter("question") | base_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt_template | rag_llm | StrOutputParser(), "context": itemgetter("context")}
)

In [68]:
base_rag_chain.invoke({"question" : "What is an agent?"})["response"]

'The provided context indicates that the term "agent" is used in various ways and lacks a clear, universally accepted definition. It is often associated with AI systems that can act on your behalf or perform tasks independently, but these interpretations are vague and sometimes conflated with concepts like autonomy or tool access. Overall, the context suggests that an "agent" generally refers to an AI system designed to act or make decisions on behalf of a user, but the precise meaning varies and remains somewhat ambiguous.'

In [69]:
base_rag_chain.invoke({"question" : "Who has produced better models than GPT-3?"})["response"]

'Organizations such as Anthropic, Mistral, Google, Meta, EleutherAI, Stability AI, TII in Abu Dhabi (Falcon), Microsoft Research, xAI, Replit, and Baidu have produced models that are better than GPT-3.'

In [70]:
base_rag_chain.invoke({"question" : "What is the laziest time of the year for AI?"})["response"]

'The provided context does not specify a particular time of year that is considered the "laziest" for AI.'

In [71]:
base_rag_chain.invoke({"question" : "What is the largest model that Simon has run on his phone?"})["response"]

'I do not know.'

### Fine-tuned Embedding Model

Now let's rebuild our RAG chain with the Fine-tuned model - the only component we need to change is our `FAISS` vectorstore!

In [72]:
finetune_vectorstore = FAISS.from_documents(training_documents, finetune_embeddings)
finetune_retriever = finetune_vectorstore.as_retriever(search_kwargs={"k": 6})

In [73]:
finetune_rag_chain = (
    {"context": itemgetter("question") | finetune_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt_template | rag_llm | StrOutputParser(), "context": itemgetter("context")}
)

In [74]:
finetune_rag_chain.invoke({"question" : "What is an Agent?"})["response"]

'An agent, in the context of AI and LLMs, is a term that lacks a clear, universally agreed-upon definition. It generally refers to systems that act on your behalf, such as travel agents or digital assistants, or to LLMs that have been given access to tools and can operate in loops to solve problems. However, the term is often used vaguely, and its precise meaning varies among different people. Overall, true autonomous agents that reliably perform meaningful tasks remain elusive, partly due to challenges like gullibility and the difficulty of distinguishing truth from fiction.'

In [75]:
finetune_rag_chain.invoke({"question" : "Who has produced better models than GPT-3?"})["response"]

'Several organizations have produced models that are better than GPT-3. According to the provided information, these include Anthropic, Mistral, Google, Meta, EleutherAI, Stability AI, TII in Abu Dhabi (Falcon), Microsoft Research, xAI, Replit, Baidu, and others.'

In [76]:
finetune_rag_chain.invoke({"question" : "What is the laziest time of the year for AI?"})["response"]

'The provided context does not specify or mention a particular time of year that is considered the "laziest" for AI. Therefore, I do not know.'

In [77]:
finetune_rag_chain.invoke({"question" : "What is the largest model that Simon has run on his phone?"})["response"]

'The largest model that Simon has run on his phone is Mistral 7B.'

#### ❓Question #2:

Which LCEL RAG Chain do you think answered the questions better, and why?

## Task 2: RAGAS Evaluation

It's great to have some idea of how our system is doing based on vibe-checks, but let's use RAGAS to provide more insight info. on how things are improving!

> NOTE: Please recreate *exactly* the RAGAS process we used to evaluate RAG, baselining with the default retriever, and then comparing the new retriever. The includes the Synthetic Data Generation steps.

### Generate Synthetic Data Set

In [78]:
pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.9 MB/s eta 0:00:00


In [79]:
pip install -qU langchain-community langchain-openai unstructured langgraph langchain-qdrant

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 48.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.4/189.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 22.9 MB/s eta 0:00:00
  

In [80]:
import asyncio
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer


Let's define our question-answer generator model.

In [81]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings
)


Now, let's define our dataset using the same query distribution as before.

In [82]:
query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]
docs = text_loader.load()

dataset = generator.generate_with_langchain_docs(docs, testset_size=10,query_distribution=query_distribution)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/26 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

### Apply RAG chains to it

Next, we define a function that feeds a particular query to the RAG, without the reference context (RAG is supposed to get that on its own).
We're going to use async function so that we can distribute evaluation later.

In [83]:
from langchain.schema.runnable.base import Runnable
from ragas.testset import TestsetSample
from typing import List,Tuple

async def get_response(evaluation:TestsetSample,
                       model:Runnable) -> Tuple[str, List[str]]:

    result = await model.ainvoke({"question":evaluation.eval_sample.user_input})

    response = result["response"]
    context = [c.page_content for c in result["context"]]

    return (response, context)

test = [d for d in dataset] # convert from generator to list for demonstration
print(f"***Query***\n{test[0].eval_sample.user_input}")
print(f"***Reference***\n{test[0].eval_sample.reference}")
r = await get_response(test[0], base_rag_chain)
print(f"***Response***\n{r[0]}")

***Query***
How does Meta’s Llama 3.3 demonstrate advancements in running GPT-4 class models on personal laptops?
***Reference***
Meta’s Llama 3.3 70B, released in December, is a GPT-4 class model that can run on a personal laptop such as a 64GB M2 MacBook Pro from 2023. This is remarkable because previously, models with GPT-4 level capabilities were thought to require datacenter-class servers with expensive GPUs. Running Llama 3.3 on a laptop, despite it taking up much of the available RAM and limiting other uses, showcases significant training and inference performance gains achieved over the past year, highlighting improvements in model efficiency.
***Response***
The provided context does not include specific information about Meta’s Llama 3.3 or how it demonstrates advancements in running GPT-4 class models on personal laptops. Therefore, I do not know.


To get the responses for the entire dataset, we async-iterate over it and apply the given model. `deepcopy` ensures we can modify the testset without affecting it.

Then we apply two models we have (base and finetuned) to it.

In [84]:
from ragas.testset import Testset
from copy import deepcopy

async def get_responses(testset:Testset, model:Runnable):
  """Return a deep-copy of the testset with response and context attached"""
  return_set = deepcopy(testset)

  L = await asyncio.gather(*[get_response(evaluation, model) for evaluation in return_set])

  for evaluation,(response,context) in zip(return_set,L):
    evaluation.eval_sample.response = response
    evaluation.eval_sample.retrieved_contexts = context

  return return_set


base_results = await get_responses(dataset, base_rag_chain)

finetune_results = await get_responses(dataset, finetune_rag_chain)


In [85]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does Meta’s Llama 3.3 demonstrate advancem...,[Voice and live camera mode are science fictio...,"Meta’s Llama 3.3 70B, released in December, is...",single_hop_specifc_query_synthesizer
1,How does Google's Gemini 1.5 Flash model compa...,[the then-new GPT-4 Turbo and $1/mTok for GPT-...,Google's Gemini 1.5 Flash model is priced at $...,single_hop_specifc_query_synthesizer
2,GitHub Spark what it is?,[pelicans: Your browser does not support the a...,GitHub announced their version of prompt-drive...,single_hop_specifc_query_synthesizer
3,Could you elaborate on Malte Ubl's perspective...,[of this gulibility. I’ve seen precious little...,"Malte Ubl from Vercel shared that initially, w...",single_hop_specifc_query_synthesizer
4,how google impact on ai model train cost and e...,[news to end the year was the release of DeepS...,Companies like Google are spending billions of...,single_hop_specifc_query_synthesizer
5,How do the challenges of gullibility in AI mod...,[<1-hop>\n\nof this gulibility. I’ve seen prec...,Gullibility in AI models remains a significant...,multi_hop_abstract_query_synthesizer
6,how live video integration with ai models like...,[<1-hop>\n\npelicans: Your browser does not su...,Live video integration with AI models emerged ...,multi_hop_abstract_query_synthesizer
7,how did advancements in large language models ...,[<1-hop>\n\nVoice and live camera mode are sci...,Advancements in large language models (LLMs) i...,multi_hop_abstract_query_synthesizer
8,why ChatGPT still best even after 1 year and h...,[<1-hop>\n\nmake things up is doing those peop...,ChatGPT remains unsurpassed even after nearly ...,multi_hop_specific_query_synthesizer
9,how amazon nova multimodal models fit in 2024 ...,[<1-hop>\n\nthe then-new GPT-4 Turbo and $1/mT...,Amazon Nova released image and video multimoda...,multi_hop_specific_query_synthesizer


In [86]:
base_results.to_pandas()

,user_input,retrieved_contexts,reference_contexts,response,reference,synthesizer_name
0,How does Meta’s Llama 3.3 demonstrate advancem...,[That same laptop that could just about run a ...,[Voice and live camera mode are science fictio...,The provided context does not include specific...,"Meta’s Llama 3.3 70B, released in December, is...",single_hop_specifc_query_synthesizer
1,How does Google's Gemini 1.5 Flash model compa...,[A year ago the single most notable example of...,[the then-new GPT-4 Turbo and $1/mTok for GPT-...,The provided context does not include specific...,Google's Gemini 1.5 Flash model is priced at $...,single_hop_specifc_query_synthesizer
2,GitHub Spark what it is?,[I’ve found myself using this a lot. I noticed...,[pelicans: Your browser does not support the a...,"Based on the provided context, GitHub Spark is...",GitHub announced their version of prompt-drive...,single_hop_specifc_query_synthesizer
3,Could you elaborate on Malte Ubl's perspective...,"[15 months later, I regret to say that we’re s...",[of this gulibility. I’ve seen precious little...,The provided documents do not mention Malte Ub...,"Malte Ubl from Vercel shared that initially, w...",single_hop_specifc_query_synthesizer
4,how google impact on ai model train cost and e...,[The legal arguments here are complex. I’m not...,[news to end the year was the release of DeepS...,The provided context does not contain specific...,Companies like Google are spending billions of...,single_hop_specifc_query_synthesizer
5,How do the challenges of gullibility in AI mod...,"[15 months later, I regret to say that we’re s...",[<1-hop>\n\nof this gulibility. I’ve seen prec...,The challenges of gullibility in AI models are...,Gullibility in AI models remains a significant...,multi_hop_abstract_query_synthesizer
6,how live video integration with ai models like...,[The GPT-4 barrier was comprehensively broken\...,[<1-hop>\n\npelicans: Your browser does not su...,Live video integration with AI models like Cha...,Live video integration with AI models emerged ...,multi_hop_abstract_query_synthesizer
7,how did advancements in large language models ...,[The GPT-4 barrier was comprehensively broken\...,[<1-hop>\n\nVoice and live camera mode are sci...,The advancements in large language models (LLM...,Advancements in large language models (LLMs) i...,multi_hop_abstract_query_synthesizer
8,why ChatGPT still best even after 1 year and h...,"[Meanwhile, it’s increasingly common for end u...",[<1-hop>\n\nmake things up is doing those peop...,The provided context does not explicitly expla...,ChatGPT remains unsurpassed even after nearly ...,multi_hop_specific_query_synthesizer
9,how amazon nova multimodal models fit in 2024 ...,[OpenAI themselves are charging 100x less for ...,[<1-hop>\n\nthe then-new GPT-4 Turbo and $1/mT...,"Based on the provided information, Amazon Nova...",Amazon Nova released image and video multimoda...,multi_hop_specific_query_synthesizer


### Evaluate results

Finally we evaluate the results using an LLM.

In [87]:
from ragas import EvaluationDataset

def to_ragas(testset:Testset) -> EvaluationDataset:
  return EvaluationDataset.from_pandas(testset.to_pandas())



In [88]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

judge_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4.1-nano")
    )

In [89]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)
metrics = [
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness(),
    ResponseRelevancy(),
    ContextEntityRecall(),
    NoiseSensitivity(),
    ]

In [90]:
base_eval = evaluate(
    dataset=to_ragas(base_results),
    metrics=metrics,
    llm=judge_llm,
    run_config=custom_run_config

    )


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

In [91]:
finetune_eval = evaluate(
    dataset=to_ragas(finetune_results),
    metrics=metrics,
    llm=judge_llm,
    run_config=custom_run_config

    )


Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[47]: AttributeError('StringIO' object has no attribute 'sentences')


### Comparing the mean scores

In [102]:
import pandas as pd
results = pd.DataFrame( [eval(str(d)) for d in [base_eval, finetune_eval]], index=["Base","Finetuned"] )
results

,context_recall,faithfulness,factual_correctness,answer_relevancy,context_entity_recall,noise_sensitivity_relevant
Base,0.9227,0.8818,0.5427,0.4192,0.2842,0.2986
Finetuned,1.0000,0.9117,0.6809,0.7544,0.3352,0.2311


The results demonstrate that all results have improved: positive metrics increased, while noise sensitivity decreased.

The largest increase is in `answer_relevancy` which measures how to-the-point a particular answer is to the query. This makes sense, since the finetuning of embedding modified the representation space to specifically pair query-document pairs more efficiently (rather than generic document-document).

We clearly see that the other metrics benefitted from this approach too.